# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [2]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    # TODO: loop up to `tries` times
    delay = 1
    for attempt in range(tries):
        r = requests.get(BASE, params=params, timeout=10)
    # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
    if r.status_code == 200:
            return pd.DataFrame(r.json()['daily'])
    # TODO:   raise on a status that is not in RETRY_STATUSES
    if r.status_code not in RETRY_STATUSES:
            r.raise_for_status()
    # TODO:   otherwise sleep (increasing each time) and try again
    time.sleep(delay)
    delay *= 2
    # TODO: raise if every attempt failed
    r.raise_for_status()
    pass

This creates the get_weather() function that can be used to pull data given dates. It loops three times - if the status code on an attempt is 200, then the attempt is successful, so it gets the data. If the status code is one of those in RETRY_STATUSES, then it sleeps for a moment and tries again to see if it works on another attempt. If the status code is not in RETRY_STATUSES, then the error is something with your own input, and it is not worth retrying, so it raises.

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [3]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
weather_range = get_weather(start='2026-09-05', end='2026-10-03')
target_dates = {'2026-09-05', '2026-10-03'}
weather_df = weather_range[weather_range['time'].isin(target_dates)].reset_index(drop=True)
print(weather_df)

used_fallback = False

         time  temperature_2m_max  precipitation_sum
0  2026-09-05                29.4                2.5
1  2026-10-03                22.0               14.7


I used the new get_weather() function to get the weather from the given date range and assigned this to a data frame. I then filtered so this data frame only contains the two dates we are interested in.

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [11]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'

try:
    weather_range = get_weather(start='2026-09-05', end='2026-10-03')
    wx = weather_range[weather_range['time'].isin(target_dates)].copy().reset_index(drop=True)
    wx['source'] = 'open-meteo'
    used_fallback = False
except Exception:
    used_fallback = True
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'

print(wx)

         time  temperature_2m_max  precipitation_sum      source
0  2026-09-05                29.4                2.5  open-meteo
1  2026-10-03                22.0               14.7  open-meteo


This tries using the get_weather() function, but if get_weather() raises, then it uses fallback data instead and indicates this in the new 'source' column. This way, the data is not empty, but a viewer can tell that it is not real either.

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [12]:
# TODO
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})
wx['date'] = pd.to_datetime(wx['date'])
wx

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


I renamed the columns in wx so they follow a more understandable convention, then ensured the date column was in the 'datetime' format.

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [13]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [14]:
# TODO
wetter_row = wx.loc[wx['precip_mm'].idxmax()]
drier_row = wx.loc[wx['precip_mm'].idxmin()]

difference = wetter_row['precip_mm'] - drier_row['precip_mm']

wetter_date = wetter_row['date'].strftime('%B %d, %Y')
drier_date = drier_row['date'].strftime('%B %d, %Y')

print(f"{wetter_date} was wetter than {drier_date} by {difference:.1f} mm")

October 03, 2026 was wetter than September 05, 2026 by 12.2 mm


This locates the rows in the table with the max and min precipitation, subtracts them to find the difference, and prints the dates and difference.

### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [15]:
# TODO
from datetime import datetime

pulled_at = datetime.now()
data_source = 'Fallback sample' if used_fallback else 'Real (open-meteo)'

print(f"Endpoint: {BASE}")
print(f"Coordinates: (38.03, -78.51)")
print(f"Date range: 2026-09-05 to 2026-10-03")
print(f"Pulled at: {pulled_at.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Status: {data_source}")

Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: (38.03, -78.51)
Date range: 2026-09-05 to 2026-10-03
Pulled at: 2026-10-12 02:41:35
Status: Real (open-meteo)


This prints information about the data pulled using the variables and other inputs from earlier. If we had needed the fallback data, it would indicate that.

### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

Merging the empty data frame on sales would still successfully produce a new chart. The column for precipitation would be empty, and if converted to zeros, it would show no precipitation for the entire season. This would lead you to believe that precipitation has no relationship with sales. Nobody would catch it because the merge is still successful, it just looks like there was no rain.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [17]:
# stretch
cities = {
    'Charlottesville': {'lat': 38.03, 'lon': -78.51, 'fallback': FALLBACK},
    'Orlando': {'lat': 28.54, 'lon': -81.38, 'fallback': FALLBACK},}

wx_2 = []

for city, info in cities.items():
    try:
        df_all = get_weather(start='2026-09-05', end='2026-10-03', lat=info['lat'], lon=info['lon'])
        df_city = df_all[df_all['time'].isin(target_dates)].copy().reset_index(drop=True)
        df_city['source'] = 'open-meteo'
    except Exception:
        df_city = info['fallback'].copy()
        df_city['source'] = 'fallback sample'

    df_city = df_city.rename(columns={
        'time': 'date',
        'temperature_2m_max': 'temp_max',
        'precipitation_sum': 'precip_mm',
    })
    df_city['date'] = pd.to_datetime(df_city['date'])
    df_city['city'] = city

    wx_2.append(df_city[['city', 'date', 'temp_max', 'precip_mm', 'source']])
print(wx_2)


[              city       date  temp_max  precip_mm      source
0  Charlottesville 2026-09-05      29.4        2.5  open-meteo
1  Charlottesville 2026-10-03      22.0       14.7  open-meteo,       city       date  temp_max  precip_mm      source
0  Orlando 2026-09-05      32.1        3.9  open-meteo
1  Orlando 2026-10-03      30.4       14.5  open-meteo]
